In [1]:
from IPython.display import display 

import base64
import json
import requests
from datetime import datetime, timedelta
from tqdm import tqdm

import pandas as pd 
pd.set_option('display.float_format', '{:.00f}'.format)
# Set pandas option to format floats as percentages with 2 decimal places
pd.set_option('display.float_format', '{:.2%}'.format)

import os 
import shutil
import numpy as np
import ast

import missingno as msno
import matplotlib.pyplot as plt


In [2]:
import sys
from pathlib import Path

try:
    # Works in Python scripts
    helper_path = Path(__file__).resolve().parent.parent / "helper"
except NameError:
    # Works in Jupyter notebooks
    helper_path = Path().resolve().parent / "helper"

sys.path.insert(0, str(helper_path))

# Now import your modules
from config import gam_info

from security_config import emplifi_key
from functions import calculate_rolling_avg_country_split, gnl_expander
import test_functions
import functions 

In [3]:
platformID = 'TTK'

# country
country_cols = ['PlaceID',	'TikTok Codes', gam_info['population_column']]
country_codes = pd.read_excel(f"../../{gam_info['lookup_file']}", 
                              sheet_name='CountryID', usecols=country_cols, keep_default_na=False )

# week 
week_tester = pd.read_excel(f"../../{gam_info['lookup_file']}", 
                            sheet_name='GAM Period', keep_default_na=False)
week_tester['w/c'] = pd.to_datetime(week_tester['w/c'])

# social media accounts
dtype_dict = {'Channel ID': 'str',
              'Linked FB Account': 'str'}
socialmedia_accounts = pd.read_excel(f"../../{gam_info['lookup_file']}", dtype=dtype_dict,
                                     sheet_name='Social Media Accounts new', keep_default_na=False)

socialmedia_accounts = socialmedia_accounts[(socialmedia_accounts['PlatformID'] == platformID)
                                            & 
                                            (socialmedia_accounts['Status'] == 'active')]
socialmedia_accounts = socialmedia_accounts.rename(columns={'Excluding UK': 'Channel Group'})

channel_ids = socialmedia_accounts['Channel ID'].unique().tolist()
formatted_channel_ids = ', '.join(f"'{channel_id}'" for channel_id in channel_ids)

# service
cols = ['ServiceID', 'Lumen']
service_codes = pd.read_excel(f"../../{gam_info['lookup_file']}", 
                              sheet_name='ServiceID', usecols=cols, keep_default_na=False )

### RUN TESTS
test_functions.test_lookup_files(country_codes, ['PlaceID'], [f"{platformID}_3_0", f"{platformID}_3_1", f"{platformID}_3_2"], 
                                 test_step="lookup files - ensuring country codes is correct")

test_functions.test_lookup_files(week_tester, ['w/c'], [f"{platformID}_3_3", f"{platformID}_3_4", f"{platformID}_3_5"], 
                                 test_step = "lookup files - ensuring week tester is correct")

test_functions.test_lookup_files(socialmedia_accounts, ['Channel ID'], [f"{platformID}_3_6", f"{platformID}_3_7", f"{platformID}_3_8"], 
                                 test_step = "lookup files - ensuring social media accounts is correct")


test_functions.test_lookup_files(service_codes, ['ServiceID', 'Lumen'], [f"{platformID}_3_9", f"{platformID}_3_10", f"{platformID}_3_11"], 
                                 test_step = "lookup files - ensuring service is correct")



✅ Test TTK_3_0 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test TTK_3_1 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test TTK_3_2 passed: No missing values in lookup.
...updating logbook...

✅ Test TTK_3_3 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test TTK_3_4 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test TTK_3_5 passed: No missing values in lookup.
...updating logbook...

✅ Test TTK_3_6 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test TTK_3_7 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test TTK_3_8 passed: No missing values in lookup.
...updating logbook...

✅ Test TTK_3_9 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test TTK_3_10 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test TTK_3_11 passed: No missing values in lookup.
...updating logbook...



# read in 

In [4]:
cols_that_must_not_be_empty = ['author', 'insights_viewers_by_country',
                               'insights_avg_time_watched', 'duration', 
                               'insights_reach', 'insights_completion_rate']
dataframes = []
storage_dir = f"../data/raw/{platformID}/post_level/"
csv_files = [f for f in os.listdir(storage_dir) if f.endswith(".csv")]
for f in csv_files:
    file_path = os.path.join(storage_dir, f)
    try:
        parts = f.replace(".csv", "").split("_")
        file_timeinfo = parts[0]
        if platformID != parts[1]:
            print('something is wrong with platformID in filename!')
        platformID = parts[1]
        profile_id = parts[2]
        week_str = parts[3]

        df = pd.read_csv(file_path)
        # test columns that must not be nan here -> move them to issues folder
        #Detect any required column that is entirely NaN
        entirely_nan_mask = df[cols_that_must_not_be_empty].isna().all(axis=0)
        entirely_nan_cols = entirely_nan_mask[entirely_nan_mask].index.tolist()
        
        if len(entirely_nan_cols) > 0:
            issues_dir = f"../data/raw/{platformID}/post_level/issues"
            os.makedirs(issues_dir, exist_ok=True)
                
            dest_path = os.path.join(issues_dir, f)
            shutil.move(file_path, dest_path)
            print(f"⚠️ Moved to issues (entirely NaN cols {entirely_nan_cols}): {f}")
            continue
            
        df["platformID"] = platformID
        df["profile_id"] = profile_id
        df["w/c"] = week_str
        
        if not df.empty:
            dataframes.append(df)
    except pd.errors.EmptyDataError:
        print(f"❌ Could not read file (empty or malformed): {f}")

# Combine all non-empty DataFrames
if dataframes:
    post_level_df = pd.concat(dataframes, ignore_index=True)
    print("✅ Combined DataFrame created.")
    display(post_level_df.head())
else:
    print("🚫 No valid data found to combine.")


✅ Combined DataFrame created.


,attachments,author,authorId,content_type,created_time,duration,id,link,media,message,...,insights_likes,insights_reach,insights_reach_engagement_rate,insights_shares,insights_video_views,insights_view_time,insights_viewers_by_country,platformID,profile_id,w/c
0,[{'image_url': 'https://p16-pu-sign-no.tiktokc...,"{'id': '3d596e6a-d239-434e-bb79-93e5d291b216',...",3d596e6a-d239-434e-bb79-93e5d291b216,post,2025-08-10T19:00:00+00:00,51,7536981698806811926,https://www.tiktok.com/@bbcnews/video/75369816...,video,Japan’s prime minister has described the count...,...,47391,499763,990.49%,662,603914,7909229,"[{'country': 'Others', 'percentage': 0.294}, {...",TTK,3d596e6a-d239-434e-bb79-93e5d291b216,2025-08-04
1,[{'image_url': 'https://p16-pu-sign-no.tiktokc...,"{'id': '3d596e6a-d239-434e-bb79-93e5d291b216',...",3d596e6a-d239-434e-bb79-93e5d291b216,post,2025-08-10T17:00:00+00:00,14,7536980939285499158,https://www.tiktok.com/@bbcnews/video/75369809...,video,A meteorite that crashed into a home in the US...,...,1043884,9793095,1099.01%,24274,11864899,115021704,"[{'country': 'PL', 'percentage': 0.025}, {'cou...",TTK,3d596e6a-d239-434e-bb79-93e5d291b216,2025-08-04
2,[{'image_url': 'https://p16-pu-sign-no.tiktokc...,"{'id': '3d596e6a-d239-434e-bb79-93e5d291b216',...",3d596e6a-d239-434e-bb79-93e5d291b216,post,2025-08-10T15:15:37+00:00,251,7536975832674372886,https://www.tiktok.com/@bbcnews/video/75369758...,video,Are planes actually crashing more often? #Plan...,...,51682,509953,1063.04%,1854,595149,9320007,"[{'country': 'AU', 'percentage': 0.012}, {'cou...",TTK,3d596e6a-d239-434e-bb79-93e5d291b216,2025-08-04
3,[{'image_url': 'https://p16-pu-sign-no.tiktokc...,"{'id': '3d596e6a-d239-434e-bb79-93e5d291b216',...",3d596e6a-d239-434e-bb79-93e5d291b216,post,2025-08-10T14:32:50+00:00,18,7536964795443039510,https://www.tiktok.com/@bbcnews/video/75369647...,video,Israeli Prime Minister Benjamin Netanyahu says...,...,83971,2521720,405.42%,4362,3232897,45198536,"[{'country': 'AU', 'percentage': 0.031}, {'cou...",TTK,3d596e6a-d239-434e-bb79-93e5d291b216,2025-08-04
4,[{'image_url': 'https://p16-pu-sign-no.tiktokc...,"{'id': '3d596e6a-d239-434e-bb79-93e5d291b216',...",3d596e6a-d239-434e-bb79-93e5d291b216,post,2025-08-10T13:00:00+00:00,67,7536900507957349654,https://www.tiktok.com/@bbcnews/video/75369005...,video,"Zara, Next and Marks & Spencer have all had ad...",...,70980,591246,1246.27%,1725,784636,9688889,"[{'country': 'SE', 'percentage': 0.015}, {'cou...",TTK,3d596e6a-d239-434e-bb79-93e5d291b216,2025-08-04


In [5]:
post_level_df = post_level_df.rename(columns={'platformID': 'PlatformID'})
# Count unique weeks per profile
week_counts = post_level_df.groupby('profile_id')['w/c'].nunique().reset_index()
week_counts.columns = ['profile_id', 'number_of_weeks']

# Optional: sort by number of weeks
week_counts = week_counts.sort_values(by='number_of_weeks', ascending=False)

# Display 
print(week_counts)

# Assuming post_level_df is already loaded and contains 'profile_id' and 'w/c'
# Create a pivot table: profiles as rows, weeks as columns
pivot_df = post_level_df.pivot_table(columns='profile_id', index='w/c', aggfunc='size')

# Convert to boolean: True = data exists, False = missing
#pivot_df = pivot_df.astype(bool)

# Visualize missing data
#msno.matrix(pivot_df)
#plt.title("Missing Weeks per TikTok Profile")
#plt.show()

                              profile_id  number_of_weeks
0   057b1686-d9f3-51fe-a129-5732678e609e               38
7   4be46cb0-a590-5e23-b3cf-ea59dd02c530               38
15  c3390a5e-42f2-4652-99ff-b903b61d8fc7               38
14  c02ca653-c3b6-4b34-b210-711e12f9eb2d               38
13  a824bd24-fa59-4039-88bf-4b152ffa2881               38
12  a53907d1-2b3d-57a1-be38-cea14a04dc0f               38
9   746ce2b5-197a-4eb6-86d2-aa2a39a021f7               38
8   54ac56ff-37ee-597e-89ec-d63de04c9df1               38
6   3d596e6a-d239-434e-bb79-93e5d291b216               38
4   34e41ced-576d-430a-b590-55d0c1d241b1               38
2   2ac83179-5aa1-4b36-9a7c-a8418f72a799               38
16  fbd019f7-9dcb-5f22-a0e0-86cfb49a5959               38
5   3c4d1098-0742-41ed-9182-f7ea05f398cd               26
1   1ea28c7a-b8c5-4e98-9c8b-456832eb4f21               24
3   2bf05522-b1ed-5751-a294-22f1d95f6cd3               22
10  80244afe-5625-509b-9839-0c5dfbc95d95               20
11  a1b31ad8-b

In [6]:

def extract_author_info(row):
    if pd.isna(row):
        return pd.Series({'id': None, 'name': None, 'url': None})

    if isinstance(row, str):
        try:
            author_dict = ast.literal_eval(row)
        except (ValueError, SyntaxError):
            return pd.Series({'id': None, 'name': None, 'url': None})
    elif isinstance(row, dict):
        author_dict = row
    else:
        return pd.Series({'id': None, 'name': None, 'url': None})

    return pd.Series({
        'id': author_dict.get('id'),
        'name': author_dict.get('name'),
        'url': author_dict.get('url')
    })

# Apply the function
post_level_df[["Channel ID", "Channel Name", "Channel URL"]] = post_level_df['author'].apply(extract_author_info)

In [7]:
post_level_df['w/c'].sort_values().unique()

array(['2025-03-31', '2025-04-07', '2025-04-14', '2025-04-21',
       '2025-04-28', '2025-05-05', '2025-05-12', '2025-05-19',
       '2025-05-26', '2025-06-02', '2025-06-09', '2025-06-16',
       '2025-06-23', '2025-06-30', '2025-07-07', '2025-07-14',
       '2025-07-21', '2025-07-28', '2025-08-04', '2025-08-11',
       '2025-08-18', '2025-08-25', '2025-09-01', '2025-09-08',
       '2025-09-15', '2025-09-22', '2025-09-29', '2025-10-06',
       '2025-10-13', '2025-10-20', '2025-10-27', '2025-11-03',
       '2025-11-10', '2025-11-17', '2025-11-24', '2025-12-01',
       '2025-12-08', '2025-12-15'], dtype=object)

# Views

In [8]:

minnie_cols_used = {'Date': 'w/c', #minnie has a day by day breakdown and then calculates the average
               'Profile ID': "Channel ID", # author['name'],
               'Profile name': "Channel Name", # author['url'],
               'Profile URL': "Channel URL", #author['id'],
               #'Post detail URL': 'link',
               #'Content ID': 'link', # splice out from link
               'Platform': 'PlatformID', 
               'Content type': 'content_type',
               'Media type': 'media',
               #'Title': '', # missing
               #'Description': '', # missing
               'Content': 'message',
               #'Link URL': '', #unclear
               'View on platform': 'link',
               'Engagements': 'insights_engagements',
               'Total reach': 'insights_reach', #but number is different? 
               'Video length (sec)': 'duration',
               'Video view count': 'insights_video_views',
               'Total video view time (sec)': 'insights_view_time',
               'Average time watched (sec)': 'insights_avg_time_watched',
               'Completion rate': 'insights_completion_rate',
              }

views_df = post_level_df[minnie_cols_used.values()]
views_df['link'] = views_df['link'].fillna('').astype(str)
views_df['content_id'] = views_df['link'].str.split('/').str[-1].str.split('?').str[0]
views_df.head()

/var/folders/wl/kjzhstq972q0cn5zv7h_jfbw0000gn/T/ipykernel_11623/3751647070.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  views_df['link'] = views_df['link'].fillna('').astype(str)
/var/folders/wl/kjzhstq972q0cn5zv7h_jfbw0000gn/T/ipykernel_11623/3751647070.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  views_df['content_id'] = views_df['link'].str.split('/').str[-1].str.split('?').str[0]


,w/c,Channel ID,Channel Name,Channel URL,PlatformID,content_type,media,message,link,insights_engagements,insights_reach,duration,insights_video_views,insights_view_time,insights_avg_time_watched,insights_completion_rate,content_id
0,2025-08-04,3d596e6a-d239-434e-bb79-93e5d291b216,BBC News,https://www.tiktok.com/@bbcnews,TTK,post,video,Japan’s prime minister has described the count...,https://www.tiktok.com/@bbcnews/video/75369816...,49501,499763,51,603914,7909229,13,7.53%,7536981698806811926
1,2025-08-04,3d596e6a-d239-434e-bb79-93e5d291b216,BBC News,https://www.tiktok.com/@bbcnews,TTK,post,video,A meteorite that crashed into a home in the US...,https://www.tiktok.com/@bbcnews/video/75369809...,1076272,9793095,14,11864899,115021704,10,24.10%,7536980939285499158
2,2025-08-04,3d596e6a-d239-434e-bb79-93e5d291b216,BBC News,https://www.tiktok.com/@bbcnews,TTK,post,video,Are planes actually crashing more often? #Plan...,https://www.tiktok.com/@bbcnews/video/75369758...,54210,509953,251,595149,9320007,16,0.34%,7536975832674372886
3,2025-08-04,3d596e6a-d239-434e-bb79-93e5d291b216,BBC News,https://www.tiktok.com/@bbcnews,TTK,post,video,Israeli Prime Minister Benjamin Netanyahu says...,https://www.tiktok.com/@bbcnews/video/75369647...,102236,2521720,18,3232897,45198536,14,29.88%,7536964795443039510
4,2025-08-04,3d596e6a-d239-434e-bb79-93e5d291b216,BBC News,https://www.tiktok.com/@bbcnews,TTK,post,video,"Zara, Next and Marks & Spencer have all had ad...",https://www.tiktok.com/@bbcnews/video/75369005...,73685,591246,67,784636,9688889,13,4.32%,7536900507957349654


In [9]:


# optional: test video length is all in seconds
print(f"number of entries: {views_df.shape}")
views_df = views_df[~views_df['insights_reach'].isna()]
print(f"number of entries that have reach: {views_df.shape}")

cols_fill_nan = ['insights_avg_time_watched', 'duration', 'insights_reach',
                 'insights_completion_rate']
views_df[cols_fill_nan] = views_df[cols_fill_nan].fillna(0)  # or any other value you'd like


number of entries: (14598, 17)
number of entries that have reach: (14598, 17)


In [10]:
# Define x and y values for each row
views_df['x1'] = 0
views_df['x2'] = views_df['insights_avg_time_watched']
views_df['x3'] = views_df['duration']
views_df['x output'] = 10

views_df['y1'] = views_df['insights_reach']
views_df['y2'] = views_df['insights_reach'] / 2
views_df['y3'] = views_df['insights_reach'] * views_df['insights_completion_rate']

def apply_quadratic_fast(row):
    x_vals = np.array([row['x1'], row['x2'], row['x3']], dtype=float)
    y_vals = np.array([row['y1'], row['y2'], row['y3']], dtype=float)

    if len(set(x_vals)) < 3 or row['x3'] == 0:
        return 100

    # Build matrix and solve
    A = np.vstack([x_vals**2, x_vals, np.ones(3)]).T
    coeffs = np.linalg.solve(A, y_vals)

    # Evaluate at x output
    return coeffs[0]*row['x output']**2 + coeffs[1]*row['x output'] + coeffs[2]

# Create new column with interpolated values
views_df['30sec_video_views'] = views_df.apply(apply_quadratic_fast, axis=1).astype(float)
views_df['completed_video_views'] = views_df['insights_completion_rate'] * views_df['insights_reach']

conditions = [
    views_df['insights_reach'] == 0,
    (views_df['completed_video_views'].round(0) > views_df['30sec_video_views'].round(0)),
    views_df['insights_reach'] < views_df['30sec_video_views'],
    views_df['duration'] == 0
]

choices = [
    views_df['insights_engagements'],
    views_df['completed_video_views'],
    views_df['insights_reach'] * 0.799,
    views_df['insights_engagements']
]

views_df['final_video_views'] = np.select(conditions, choices, 
                                            default=views_df['30sec_video_views'])

In [11]:
'''views_df[
    (views_df['Channel ID'] == 'c02ca653-c3b6-4b34-b210-711e12f9eb2d') &
    (views_df['w/c'] == '2025-08-04') ].sort_values('final_video_views', ascending=False)'''

"views_df[\n    (views_df['Channel ID'] == 'c02ca653-c3b6-4b34-b210-711e12f9eb2d') &\n    (views_df['w/c'] == '2025-08-04') ].sort_values('final_video_views', ascending=False)"

In [12]:
views_df_full = views_df.merge(socialmedia_accounts[['Channel ID', 'ServiceID', 'Linked FB Account']],
               on='Channel ID', how='left')
test_functions.test_inner_join(views_df, socialmedia_accounts[['Channel ID', 'ServiceID', 'Linked FB Account']], 
                               ['Channel ID'], 
                               f"12_{platformID}_engagements", 
                               test_step='checking social media accounts in lookup, adding service',
                               focus='left')


✅ Inner join test 12_TTK_engagements successful: No issues found.
...updating logbook...



In [13]:
file_path = f"../data/processed/{platformID}"
os.makedirs(file_path, exist_ok=True)

cols = ['content_id', 'ServiceID', 'Channel ID', 'Channel Name', 'w/c', 
        'link',
        'final_video_views', 'Linked FB Account'
       ]
views_df_full = views_df_full[cols]
views_df_full.to_csv(f"{file_path}/{gam_info['file_timeinfo']}_{platformID}_views.csv",
                       index=None)


In [14]:
# YT views per viewer is missing for media action 
yt_views_per_viewer = pd.read_excel("../data/stale/YT views per viewer_TTKhelper.xlsx")[['w/c', 'Service', 'views_per_viewer']]
yt_views_per_viewer = yt_views_per_viewer.rename(columns={'Service': 'Lumen'})
yt_views_per_viewer = yt_views_per_viewer.merge(service_codes, on='Lumen', how='left').drop(columns='Lumen')
yt_views_per_viewer = gnl_expander(yt_views_per_viewer)
yt_views_per_viewer.sample()

,w/c,views_per_viewer,ServiceID
3197,2024-03-25,199.70%,AZE


In [15]:
views_df_full['w/c'] = pd.to_datetime(views_df_full['w/c'])
yt_views_per_viewer['w/c'] = pd.to_datetime(yt_views_per_viewer['w/c'])

views_df_yt = views_df_full.merge(yt_views_per_viewer, on=['ServiceID', 'w/c'], how='left', indicator=True)

matched = views_df_yt[views_df_yt['_merge'] == 'both'].drop(columns='_merge')
unmatched = views_df_yt[views_df_yt['_merge'] == 'left_only'].drop(columns=['_merge', 'views_per_viewer'])

views_per_viewer_by_service = yt_views_per_viewer.groupby(['ServiceID'])['views_per_viewer'].mean().reset_index()

matched_sec = unmatched.merge(views_per_viewer_by_service, on='ServiceID', how='left', indicator=True)
matched_sec = matched_sec[matched_sec['_merge'] == 'both'].drop(columns='_merge')

views_scaled = pd.concat([matched, matched_sec]).dropna(subset=['final_video_views'])
views_scaled = views_scaled[views_scaled['final_video_views'] > 0]
views_scaled['engaged_users'] = views_scaled['final_video_views']/(views_scaled['views_per_viewer']*1.14)

views_scaled.columns

Index(['content_id', 'ServiceID', 'Channel ID', 'Channel Name', 'w/c', 'link',
       'final_video_views', 'Linked FB Account', 'views_per_viewer',
       'engaged_users'],
      dtype='object')

# Country 

In [16]:
country_df = post_level_df.copy()

In [17]:

# Step 1: Parse the stringified list of country-percentage dictionaries
country_df['parsed_viewers'] = country_df['insights_viewers_by_country'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else []
)

# Step 2: Explode the parsed list into separate rows
exploded_df = country_df.explode('parsed_viewers').reset_index(drop=True)

# Step 3: Extract 'country' and 'percentage' from each dictionary
exploded_df[['country', 'percentage']] = exploded_df['parsed_viewers'].apply(
    lambda entry: pd.Series({
        'country': entry.get('country') if isinstance(entry, dict) else None,
        'percentage': entry.get('percentage') if isinstance(entry, dict) else None
    })
)

# Step 4: Drop the intermediate column
exploded_df.drop(columns=['parsed_viewers'], inplace=True)
exploded_df['country'] = exploded_df['country'].replace('Others', 'ZZ')
exploded_df['country'] = exploded_df['country'].fillna('ZZ')


In [18]:

exploded_df = exploded_df.rename(columns={'country': 'TikTok Codes'})
ttk_country_all = exploded_df.merge(country_codes[['TikTok Codes', 'PlaceID']], on='TikTok Codes', how='left',
                 indicator=True)

print(f"mismatches? \n{ttk_country_all._merge.value_counts()}")
ttk_country_all = ttk_country_all.drop(columns='_merge')

# Remove unknown countries (UNK)
ttk_country = ttk_country_all[ttk_country_all['PlaceID'] != 'UNK'].copy()

# Compute sum of percentages per video
sum_per_video = ttk_country.groupby('id')['percentage'].transform('sum')

# Rescale percentages
ttk_country['rescaled_percentage'] = ttk_country['percentage'] / sum_per_video

# Optional: Check that each video sums to 1
check = ttk_country.groupby('id')['rescaled_percentage'].sum()
check[~np.isclose(check, 1, atol=1e-6)]

country_cols = ['Channel ID', 'link', 'PlaceID', 'rescaled_percentage', 'w/c', 'PlatformID']
ttk_country= ttk_country[country_cols]
ttk_country.to_csv(f"../data/processed/{platformID}/{gam_info['file_timeinfo']}_{platformID}_country.csv",
                  index=None, na_rep='')

mismatches? 
_merge
both          157164
left_only          0
right_only         0
Name: count, dtype: int64


In [19]:
# Weekly mean per channel/place to get to a rolling channel / coutnry split average
ttk_country_avg_channel = (
    ttk_country
    .groupby(['Channel ID', 'w/c', 'PlaceID', 'PlatformID'])['rescaled_percentage']
    .mean()
    .reset_index()
)

avg_country_df = calculate_rolling_avg_country_split(ttk_country_avg_channel, 'rescaled_percentage',
                                                     ttk_country['w/c'].min(), ttk_country['w/c'].max())



# combine views & country

country is a percentage by video. In the next section the actual metric is calculated (video views) per country. This is then summed to the profile level. 
As the average is larger than 1 the country view is rebased 


In [20]:
# 1. Convert week column to datetime
ttk_country['w/c'] = pd.to_datetime(ttk_country['w/c'])

# 2. Remove merge indicators if present
ttk_country = ttk_country.drop(columns=['_merge'], errors='ignore')
views_scaled = views_scaled.drop(columns=['_merge'], errors='ignore')

# 3. Initial merge: country-level data with scaled views
merged_initial = ttk_country.merge(
    views_scaled,
    on=['Channel ID', 'link', 'w/c'],
    how='outer',
    indicator=True
)
print(f"Initial merge mismatches:\n{merged_initial._merge.value_counts()}")

# deal with country
unmatched_country = merged_initial[merged_initial['_merge'] == 'left_only']
# for each channel / w/c combination I want the country average of the previous 4 weeks to be the fill in for the missing data
merged_country_avg = avg_country_df[['w/c', 'Channel ID', 'PlaceID', 'rescaled_percentage']].merge(unmatched_country[views_scaled.columns],
    on=['Channel ID', 'w/c'],
    how='outer',
    indicator=True
)
print(f"Country second merge mismatches:\n{merged_country_avg._merge.value_counts()}")
# left only is fine these are weeky percentages in weeks that don't have uv data

# 4. Identify unmatched rows (right_only - country only)
unmatched_views = merged_initial[merged_initial['_merge'] == 'right_only']

# 5. Compute weekly averages for unmatched rows
weekly_avg = ttk_country.groupby(['Channel ID', 'w/c'])['rescaled_percentage'].mean().reset_index()
merged_weekly_avg = weekly_avg.merge(
    unmatched_views[views_scaled.columns],
    on=['Channel ID', 'w/c'],
    how='outer',
    indicator=True
)
print(f"Mismatches after weekly average merge:\n{merged_weekly_avg._merge.value_counts()}")

# 6. Identify remaining unmatched rows and compute channel-level averages
still_unmatched = merged_weekly_avg[merged_weekly_avg['_merge'] == 'right_only']
channel_avg = ttk_country.groupby(['Channel ID'])['rescaled_percentage'].mean().reset_index()
merged_channel_avg = channel_avg.merge(
    still_unmatched[views_scaled.columns],
    on=['Channel ID'],
    how='outer',
    indicator=True
)
print(f"Mismatches after channel-level average merge:\n{merged_channel_avg._merge.value_counts()}")

# 7. Combine all data sources (same as original logic)
combined_data = pd.concat(
    [merged_initial, merged_country_avg, merged_weekly_avg, merged_channel_avg],
    ignore_index=True
).drop(columns=['_merge'])

# 8. Calculate country-level views at video level
combined_data['country_views_video_level'] = (
    combined_data['final_video_views'] * combined_data['rescaled_percentage']
)

# 9. add population column 
combined_data = combined_data.merge(country_codes[['PlaceID', gam_info['population_column']]],
                                    on=['PlaceID'], how='left')


Initial merge mismatches:
_merge
both          142092
left_only       1409
right_only       319
Name: count, dtype: int64
Country second merge mismatches:
_merge
both          58837
left_only     31464
right_only        0
Name: count, dtype: int64
Mismatches after weekly average merge:
_merge
left_only     443
both          319
right_only      0
Name: count, dtype: int64
Mismatches after channel-level average merge:
_merge
left_only     17
right_only     0
both           0
Name: count, dtype: int64


In [21]:
merged_country_avg[merged_country_avg['_merge'] == 'left_only']

,w/c,Channel ID,PlaceID,rescaled_percentage,content_id,ServiceID,Channel Name,link,final_video_views,Linked FB Account,views_per_viewer,engaged_users,_merge
0,2025-03-31,057b1686-d9f3-51fe-a129-5732678e609e,ALG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1,2025-03-31,057b1686-d9f3-51fe-a129-5732678e609e,AUS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2,2025-03-31,057b1686-d9f3-51fe-a129-5732678e609e,BEL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
3,2025-03-31,057b1686-d9f3-51fe-a129-5732678e609e,BRN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
4,2025-03-31,057b1686-d9f3-51fe-a129-5732678e609e,BUR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...,...,...,...,...,...
90296,2025-12-15,fbd019f7-9dcb-5f22-a0e0-86cfb49a5959,THA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
90297,2025-12-15,fbd019f7-9dcb-5f22-a0e0-86cfb49a5959,UAE,10.32%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
90298,2025-12-15,fbd019f7-9dcb-5f22-a0e0-86cfb49a5959,UK,3.44%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
90299,2025-12-15,fbd019f7-9dcb-5f22-a0e0-86cfb49a5959,UKR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [22]:
# 10. Apply Sainsbury formula for country-level views
deduplicated_datasets = []
for channel in tqdm(combined_data['Channel ID'].unique()):
    temp = combined_data[combined_data['Channel ID'] == channel]
    channel_uv_by_country = pd.crosstab(
                                        index = [ temp['PlaceID'], 
                                                  temp['PlatformID'],
                                                  temp['ServiceID'],
                                                  temp['Channel ID'],
                                                  temp[gam_info['population_column']],
                                                  temp['w/c']],
                                        columns = temp['link'],
                                        values =  temp['country_views_video_level'],
                                        aggfunc='sum'
                                    )
    link_ids = channel_uv_by_country.columns
    channel_uv_by_country = channel_uv_by_country.reset_index().fillna(0)
    channel_uv_by_country = functions.sainsbury_formula(channel_uv_by_country, 
                                                        gam_info['population_column'],
                                                        link_ids, 
                                                        'country_views_video_level')
    channel_uv_by_country = channel_uv_by_country.drop(columns=link_ids)
    deduplicated_datasets.append(channel_uv_by_country)

dedupli_df = pd.concat(deduplicated_datasets)
        

100%|█████████████████████████████████████████████████████████████████████| 17/17 [00:11<00:00,  1.45it/s]


In [23]:
# 11. Aggregate to profile level (country-specific)
country_profile_views = dedupli_df.groupby(
    ['w/c', 'Channel ID', 'ServiceID', 'PlaceID']
)['country_views_video_level'].sum().rename('country_views_profile_level').reset_index()

# 12. Aggregate to global profile level
global_profile_views = dedupli_df.groupby(
    ['w/c', 'ServiceID', 'Channel ID']
)['country_views_video_level'].sum().rename('global_views_profile_level').reset_index()

# 13. Merge country and global profile views
profile_views = country_profile_views.merge(
    global_profile_views,
    on=['ServiceID', 'w/c', 'Channel ID'],
    how='outer',
    indicator=True
)
print(f"Profile-level merge check:\n{profile_views._merge.value_counts()}")
profile_views = profile_views.drop(columns=['_merge'])

# 14. Calculate percentage contribution of country views
profile_views['profile_country_views_%'] = (
    profile_views['country_views_profile_level'] / profile_views['global_views_profile_level']
)

# 15. Merge back with scaled views for user-level metrics
ttk_df = profile_views.merge(
    views_scaled,
    on=['ServiceID', 'Channel ID', 'w/c'],
    how='inner'
)

# 16. Calculate UV by country
ttk_df['uv_by_country'] = (
    ttk_df['engaged_users'] * ttk_df['profile_country_views_%']
)


Profile-level merge check:
_merge
both          10796
left_only         0
right_only        0
Name: count, dtype: int64


In [24]:
print(ttk_df.shape)
ttk_df = ttk_df.dropna(subset='uv_by_country')
print(ttk_df.shape)

cols = ['w/c', 'PlaceID', 'ServiceID', 'Channel ID', 'uv_by_country', ]
ttk_df[cols].to_csv(f"../data/processed/{platformID}/{gam_info['file_timeinfo']}_{platformID}_uniqueViewer_country.csv", 
                     index=None)

(426552, 15)
(426552, 15)


In [25]:
ttk_df[ttk_df['w/c'] == '2025-12-01']['ServiceID'].unique()

array(['ARA', 'AZE', 'BEN', 'BNO', 'ELT', 'INO', 'PDG', 'POR', 'RUS',
       'SPA', 'SWA', 'THA', 'TUR', 'UKR', 'URD'], dtype=object)

In [26]:
ttk_df[ttk_df['w/c'] == '2025-11-24']['ServiceID'].unique()

array(['ARA', 'AZE', 'BEN', 'BNO', 'ELT', 'INO', 'PDG', 'POR', 'RUS',
       'SPA', 'SWA', 'THA', 'TUR', 'UKR', 'URD'], dtype=object)

In [27]:
ttk_df['w/c'].sort_values().unique()

<DatetimeArray>
['2025-03-31 00:00:00', '2025-04-07 00:00:00', '2025-04-14 00:00:00',
 '2025-04-21 00:00:00', '2025-04-28 00:00:00', '2025-05-05 00:00:00',
 '2025-05-12 00:00:00', '2025-05-19 00:00:00', '2025-05-26 00:00:00',
 '2025-06-02 00:00:00', '2025-06-09 00:00:00', '2025-06-16 00:00:00',
 '2025-06-23 00:00:00', '2025-06-30 00:00:00', '2025-07-07 00:00:00',
 '2025-07-14 00:00:00', '2025-07-21 00:00:00', '2025-07-28 00:00:00',
 '2025-08-04 00:00:00', '2025-08-11 00:00:00', '2025-08-18 00:00:00',
 '2025-08-25 00:00:00', '2025-09-01 00:00:00', '2025-09-08 00:00:00',
 '2025-09-15 00:00:00', '2025-09-22 00:00:00', '2025-09-29 00:00:00',
 '2025-10-06 00:00:00', '2025-10-13 00:00:00', '2025-10-20 00:00:00',
 '2025-10-27 00:00:00', '2025-11-03 00:00:00', '2025-11-10 00:00:00',
 '2025-11-17 00:00:00', '2025-11-24 00:00:00', '2025-12-01 00:00:00',
 '2025-12-08 00:00:00', '2025-12-15 00:00:00']
Length: 38, dtype: datetime64[ns]